# Chunking sweep: chunk size vs retrieval quality

The deployed index uses 1000-character chunks with recursive splitting. This notebook 
re-chunks the same 484 ClinicalTrials.gov records at 500 and 1500 characters, embeds 
each set with PubMedBERT, builds a FAISS index per size, and compares retrieval on 
a held-out query set spanning four categories (specific drug, biomarker, modality, 
adversarial OOD).

Goal: evidence-backed answer for the README on which chunk size ships.

In [2]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

sys.path.append("..")
from src.rag import retrieve, EMBEDDING_MODEL

DATA_DIR = Path("../data")
SWEEP_DIR = DATA_DIR / "sweep"
SWEEP_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 80)

In [3]:
# The deployed pipeline saved cleaned trials before chunking. Reuse them
# so the sweep is a fair comparison (same input, only chunk size varies).

with open(DATA_DIR / "trials.pkl", "rb") as f:
    trials = pickle.load(f)

print(f"Loaded {len(trials)} trials")
print(f"Container: {type(trials).__name__}")

sample = trials[0] if isinstance(trials, list) else trials.iloc[0]
fields = list(sample.keys()) if hasattr(sample, "keys") else sample.index.tolist()
print(f"Fields: {fields}")

# Show a small preview so we know which field holds the long text body
for f in fields:
    val = sample[f]
    preview = str(val)[:60].replace("\n", " ")
    print(f"  {f:20s} -> {preview!r}  (len={len(str(val))})")

Loaded 484 trials
Container: list
Fields: ['nct_id', 'title', 'text', 'conditions']
  nct_id               -> 'NCT05486988'  (len=11)
  title                -> 'ctDNA as a Biomarker for Treatment in Advanced NSCLC'  (len=52)
  text                 -> 'Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC '  (len=2417)
  conditions           -> "['Non-small Cell Lung Cancer']"  (len=30)


## Step A — Re-chunk at 500 and 1500

Same source trials, same recursive splitter (`["\n\n", "\n", " ", ""]`), same overlap 
(100 chars) as the production 1000-char build. Only chunk size varies. Fixed absolute 
overlap rather than proportional — this is the convention in the LangChain ecosystem 
and what most published RAG comparisons use. Worth noting in the README.

In [4]:
def chunk_trials(trials, chunk_size, overlap=100):
    """Re-chunk all trials at a given chunk size. Returns list of dicts
    matching the production schema: {nct_id, title, text}."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function=len,
    )

    chunks = []
    for trial in trials:
        pieces = splitter.split_text(trial["text"])
        for piece in pieces:
            chunks.append({
                "nct_id": trial["nct_id"],
                "title": trial["title"],
                "text": piece,
            })
    return chunks


chunks_500 = chunk_trials(trials, chunk_size=500)
chunks_1500 = chunk_trials(trials, chunk_size=1500)

print(f"  500-char chunks: {len(chunks_500):,}  (avg {len(chunks_500)/len(trials):.1f} per trial)")
print(f" 1000-char chunks: 3,264 (production, on disk)")
print(f" 1500-char chunks: {len(chunks_1500):,}  (avg {len(chunks_1500)/len(trials):.1f} per trial)")

  500-char chunks: 6,783  (avg 14.0 per trial)
 1000-char chunks: 3,264 (production, on disk)
 1500-char chunks: 2,141  (avg 4.4 per trial)


In [5]:
# Save so we can reuse without re-chunking. Embedding is the slow step.
with open(SWEEP_DIR / "chunks_500.pkl", "wb") as f:
    pickle.dump(chunks_500, f)
with open(SWEEP_DIR / "chunks_1500.pkl", "wb") as f:
    pickle.dump(chunks_1500, f)

# Sanity: a chunk from each size, same trial, side by side
sample_nct = chunks_500[0]["nct_id"]

print(f"=== {sample_nct} — first 500-char chunk ===")
print(chunks_500[0]["text"])
print()
print(f"=== {sample_nct} — first 1500-char chunk ===")
print(next(c for c in chunks_1500 if c["nct_id"] == sample_nct)["text"])

=== NCT05486988 — first 500-char chunk ===
Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC

Conditions: Non-small Cell Lung Cancer

Brief Summary: The dynamic monitoring of circulating tumor DNA aims to evaluate the response and progression-free survival of short-course chemotherapy (2 cycles) combined with immunotherapy in patients with locally advanced unresectable or metastatic non-small cell lung cancer.

=== NCT05486988 — first 1500-char chunk ===
Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC

Conditions: Non-small Cell Lung Cancer

Brief Summary: The dynamic monitoring of circulating tumor DNA aims to evaluate the response and progression-free survival of short-course chemotherapy (2 cycles) combined with immunotherapy in patients with locally advanced unresectable or metastatic non-small cell lung cancer.

Detailed Description: For patients with locally advanced unresectable or metastatic non-small cell lung cancers, 4-6 cycles of chemotherapy plus immu

## Step B — Embed and index

PubMedBERT (`pritamdeka/S-PubMedBert-MS-MARCO`), 768-dim, normalized embeddings.
FAISS `IndexFlatL2` to match production. With normalized vectors, L2 distance and
cosine similarity are monotonically related, so this gives identical ranking to
an inner-product index — same conversion `cosine_sim = 1 - dist/2` as `src/rag.py`.

In [ ]:
#Cell 8 (load model)
# Same model the production index was built with. SentenceTransformer caches
# it locally after first download, so this should be instant on second run.
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Model loaded: {EMBEDDING_MODEL}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Device: {model.device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: pritamdeka/S-PubMedBert-MS-MARCO
Embedding dim: 768
Device: cpu


C:\Users\shrik\AppData\Local\Temp\ipykernel_15728\2210342819.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")


In [ ]:
#Cell 9 (embed + index, 500-char)
texts_500 = [c["text"] for c in chunks_500]

embeddings_500 = model.encode(
    texts_500,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

index_500 = faiss.IndexFlatL2(embeddings_500.shape[1])
index_500.add(embeddings_500)

print(f"Embeddings: {embeddings_500.shape}")
print(f"Index size: {index_500.ntotal} vectors")

Batches:   0%|          | 0/212 [00:00<?, ?it/s]

Embeddings: (6783, 768)
Index size: 6783 vectors


In [8]:
#Cell 10 (embed + index, 1500-char)
texts_1500 = [c["text"] for c in chunks_1500]

embeddings_1500 = model.encode(
    texts_1500,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

index_1500 = faiss.IndexFlatL2(embeddings_1500.shape[1])
index_1500.add(embeddings_1500)

print(f"Embeddings: {embeddings_1500.shape}")
print(f"Index size: {index_1500.ntotal} vectors")

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

Embeddings: (2141, 768)
Index size: 2141 vectors


In [9]:
#Cell 11 (persist + load production 1000 baseline)
# Save sweep artifacts
faiss.write_index(index_500,  str(SWEEP_DIR / "faiss_500.index"))
faiss.write_index(index_1500, str(SWEEP_DIR / "faiss_1500.index"))
np.save(SWEEP_DIR / "embeddings_500.npy",  embeddings_500)
np.save(SWEEP_DIR / "embeddings_1500.npy", embeddings_1500)

# Load production 1000-char baseline so all three are in memory together
with open(DATA_DIR / "chunks.pkl", "rb") as f:
    chunks_1000 = pickle.load(f)
index_1000 = faiss.read_index(str(DATA_DIR / "faiss.index"))

print(f"  500: {len(chunks_500):>5,} chunks | index ntotal = {index_500.ntotal:,}")
print(f" 1000: {len(chunks_1000):>5,} chunks | index ntotal = {index_1000.ntotal:,}")
print(f" 1500: {len(chunks_1500):>5,} chunks | index ntotal = {index_1500.ntotal:,}")

  500: 6,783 chunks | index ntotal = 6,783
 1000: 3,264 chunks | index ntotal = 3,264
 1500: 2,141 chunks | index ntotal = 2,141


## Step C — Query set + retrieval comparison

Seven queries probing the dimensions where chunk size theoretically matters:
- Q1-Q2: in-corpus drug/biomarker baselines (sanity check, all sizes should perform well)
- Q3: known limitation from Block 4 (ADC ↔ T-DXd vocabulary mismatch)
- Q4-Q5: queries needing content typically deeper in trial text (stats, eligibility)
- Q6-Q7: adversarial — out-of-domain and in-domain-vocabulary OOD

Same retrieval logic as production (`src.rag.retrieve`, k=5, fetch_multiplier=4, 
dedup by nct_id). Only chunk size and index change.

In [ ]:
#Cell 13 (query set)
TEST_QUERIES = [
    {"query": "pembrolizumab in non-small cell lung cancer",
     "category": "drug + condition",
     "expect": "in-corpus, multi-concept, should rank well at all sizes"},

    {"query": "BRAF V600E mutation targeted therapy",
     "category": "biomarker baseline",
     "expect": "known to retrieve well per Block 3 testing"},

    {"query": "antibody drug conjugate for HER2 positive cancer",
     "category": "vocabulary mismatch",
     "expect": "Block 4 found T-DXd chunks present but missed; test if 1500 helps"},

    {"query": "hazard ratio for overall survival in phase 3 trials",
     "category": "deeper context",
     "expect": "stats numbers usually appear deeper in trial body; 1500 may capture better"},

    {"query": "trial eligibility criteria age 65 and older",
     "category": "deeper context",
     "expect": "eligibility section is typically further down the trial text"},

    {"query": "best Italian restaurant in Boston",
     "category": "adversarial OOD",
     "expect": "should be refused at threshold by all sizes"},

    {"query": "treatment for the common cold",
     "category": "adversarial in-domain vocab",
     "expect": "medical-sounding but not oncology; threshold should refuse"},
]

print(f"{len(TEST_QUERIES)} queries across {len(set(q['category'] for q in TEST_QUERIES))} categories")

7 queries across 6 categories


In [11]:
#Cell 14 (retrieval helpers)
def retrieve_all_sizes(query, k=5):
    """Run production retrieve() against all three indices."""
    return {
        500:  retrieve(query, model, index_500,  chunks_500,  k=k),
        1000: retrieve(query, model, index_1000, chunks_1000, k=k),
        1500: retrieve(query, model, index_1500, chunks_1500, k=k),
    }

def compare_query(query_obj, k=5, show_text_chars=250):
    """Print top-3 retrieved chunks from each size, side by side."""
    q = query_obj["query"]
    print("=" * 90)
    print(f"QUERY: {q}")
    print(f"  Category: {query_obj['category']}")
    print(f"  Expect:   {query_obj['expect']}")
    print("=" * 90)

    results_by_size = retrieve_all_sizes(q, k=k)
    for size in [500, 1000, 1500]:
        r_list = results_by_size[size]
        print(f"\n--- {size}-char chunks ---  top-1 sim = {r_list[0]['score']:.3f}")
        for r in r_list[:3]:
            preview = r["text"][:show_text_chars].replace("\n", " ")
            print(f"  #{r['rank']} [{r['nct_id']}] sim={r['score']:.3f}")
            print(f"      {preview}...")
    print()
    return results_by_size

In [12]:
#Cell 15 (run all queries — manual review)
all_results = {}
for q_obj in TEST_QUERIES:
    all_results[q_obj["query"]] = compare_query(q_obj, k=5, show_text_chars=250)

QUERY: pembrolizumab in non-small cell lung cancer
  Category: drug + condition
  Expect:   in-corpus, multi-concept, should rank well at all sizes

--- 500-char chunks ---  top-1 sim = 0.962
  #1 [NCT04340882] sim=0.962
      compared to docetaxel alone in this setting. Pembrolizumab is an FDA-approved treatment for NSCLC and can be given alone or in combination with platinum-based chemotherapy....
  #2 [NCT05418660] sim=0.958
      Detailed Description: Programmed cell Death protein / Ligand 1 (PD-1 / PD-L1) inhibitors Atezolizumab, Nivolumab and Pembrolizumab have demonstrated a great efficacy and a good safety profile in patients with Non-Small Cell Lung Cancer (NSCLC), both ...
  #3 [NCT02991482] sim=0.954
      Title: PembROlizuMab Immunotherapy Versus Standard Chemotherapy for Advanced prE-treated Malignant Pleural Mesothelioma  Conditions: Pleural Mesothelioma Malignant Advanced  Brief Summary: Trial comparing standard treatment (chemotherapy) with pembro...

--- 1000-char chun

In [13]:
#Cell 16 (numeric summary table)
rows = []
for q_obj in TEST_QUERIES:
    q = q_obj["query"]
    res = all_results[q]
    rows.append({
        "category":      q_obj["category"],
        "query":         q[:48],
        "top1_500":      round(res[500][0]["score"],  3),
        "top1_1000":     round(res[1000][0]["score"], 3),
        "top1_1500":     round(res[1500][0]["score"], 3),
        "nct_500":       res[500][0]["nct_id"],
        "nct_1000":      res[1000][0]["nct_id"],
        "nct_1500":      res[1500][0]["nct_id"],
    })

summary = pd.DataFrame(rows)
summary["best_size"] = summary[["top1_500", "top1_1000", "top1_1500"]].idxmax(axis=1).str.replace("top1_", "")
summary["all_agree_on_top"] = summary.apply(
    lambda r: r["nct_500"] == r["nct_1000"] == r["nct_1500"], axis=1
)
summary.to_csv(SWEEP_DIR / "chunking_sweep_summary.csv", index=False)
summary

,category,query,top1_500,top1_1000,top1_1500,nct_500,nct_1000,nct_1500,best_size,all_agree_on_top
0,drug + condition,pembrolizumab in non-small cell lung cancer,0.962,0.954,0.954,NCT04340882,NCT02991482,NCT02991482,500,False
1,biomarker baseline,BRAF V600E mutation targeted therapy,0.946,0.938,0.931,NCT02414750,NCT04151563,NCT02414750,500,False
2,vocabulary mismatch,antibody drug conjugate for HER2 positive cancer,0.937,0.932,0.930,NCT06727227,NCT00004888,NCT00004888,500,False
3,deeper context,hazard ratio for overall survival in phase 3 tri,0.939,0.922,0.921,NCT02224781,NCT03568097,NCT03568097,500,False
4,deeper context,trial eligibility criteria age 65 and older,0.945,0.945,0.939,NCT05353686,NCT05353686,NCT06707220,500,False
5,adversarial OOD,best Italian restaurant in Boston,0.834,0.834,0.835,NCT06727227,NCT07342010,NCT06005142,1500,False
6,adversarial in-domain vocab,treatment for the common cold,0.905,0.903,0.898,NCT00656227,NCT07513883,NCT07513883,500,False


## Step 2 — RAGAS evaluation

Three reference-free metrics across 12 queries × 3 chunk sizes (36 RAG runs):

- **Faithfulness**: does the generated answer only use claims supported by retrieved 
  context? (LLM judge decomposes answer into claims, checks each against context.)
- **Answer relevancy**: does the answer address the question? (Reverse-question 
  embedding similarity.)
- **Context precision**: are the retrieved chunks actually relevant to the question? 
  (LLM judge per chunk.)

LLM judge: Groq Llama 3.3 70B (same model used for generation — fine for relative 
comparison across chunk sizes; we're measuring deltas, not absolute truth).

In [16]:
# Install if not already present. Comment out if installed.
# !pip install ragas==0.2.10 langchain-groq==0.2.0 datasets

import ragas
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset
import os

print(f"RAGAS version: {ragas.__version__}")
print(f"GROQ_API_KEY set: {bool(os.getenv('GROQ_API_KEY'))}")

ImportError: cannot import name '_LC_ID_PREFIX' from 'langchain_core.messages.ai' (c:\Users\shrik\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_core\messages\ai.py)

## Step 2 — Hand-rolled RAGAS-equivalent evaluation

Three metrics, computed with Groq Llama 3.3 70B as judge:

- **Faithfulness**: % of atomic claims in the answer that are supported by the
  retrieved context. Decompose answer into claims, judge each one.
- **Answer relevancy**: cosine similarity between the original question and a 
  question reverse-generated from the answer. High score = answer addresses 
  the question. Low score = answer drifts.
- **Context precision**: % of retrieved chunks that are actually relevant to 
  the question. Judge each chunk independently.

6 in-corpus queries × 3 chunk sizes = 18 RAG runs. Adversarial queries reported
separately as refusal rate, since faithfulness/relevancy are undefined for refusals.

In [17]:
#Cell 20 (judge wrapper + helpers)
from src.rag import generate, build_context, make_groq_client, retrieve, LLM_MODEL
import json
import re
import time

client = make_groq_client()

def judge(prompt, max_tokens=300, temperature=0.0):
    """Single Groq call returning text. Used for all metric judging."""
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

def parse_json_list(text):
    """Pull a JSON list out of LLM output, tolerant of code-fence wrapping."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return []

def parse_yes_no(text):
    """Return 1 for yes, 0 for no, None if unclear."""
    text = text.strip().lower()
    if text.startswith("yes"):  return 1
    if text.startswith("no"):   return 0
    if "yes" in text[:20]:      return 1
    if "no"  in text[:20]:      return 0
    return None

In [18]:
#Cell 21 (eval query set)
EVAL_QUERIES = [
    "What trials use pembrolizumab in non-small cell lung cancer?",
    "Which trials target the BRAF V600E mutation?",
    "Are there trials studying antibody drug conjugates for HER2 positive cancer?",
    "What are reported hazard ratios for overall survival in phase 3 oncology trials?",
    "What are the eligibility criteria for trials enrolling patients age 65 and older?",
    "What trials use CAR-T cell therapy for leukemia or lymphoma?",
]

ADVERSARIAL_QUERIES = [
    "best Italian restaurant in Boston",
    "treatment for the common cold",
    "how do I file my taxes",
    "what is the capital of France",
]

print(f"In-corpus eval queries: {len(EVAL_QUERIES)}")
print(f"Adversarial queries:    {len(ADVERSARIAL_QUERIES)}")

In-corpus eval queries: 6
Adversarial queries:    4


In [20]:
#Cell 22 (run RAG on all combos and persist)
INDICES_BY_SIZE = {500: index_500,  1000: index_1000, 1500: index_1500}
CHUNKS_BY_SIZE  = {500: chunks_500, 1000: chunks_1000, 1500: chunks_1500}

rag_runs = []   # list of dicts with query, size, answer, contexts, top_score, refused_at
t0 = time.time()

for q in EVAL_QUERIES + ADVERSARIAL_QUERIES:
    for size in [500, 1000, 1500]:
        result = generate(q, client, model, INDICES_BY_SIZE[size], CHUNKS_BY_SIZE[size], k=5)
        rag_runs.append({
            "query":       q,
            "size":        size,
            "answer":      result["answer"],
            "contexts":    [r["text"] for r in result["sources"]],
            "context_ncts":[r["nct_id"] for r in result["sources"]],
            "top_score":   result["top_score"],
            "refused_at":  result["refused_at"],
            "is_adversarial": q in ADVERSARIAL_QUERIES,
        })
        print(f"  {size:>4} | {q[:55]:55s} | refused={result['refused_at']}")

print(f"\nTotal: {len(rag_runs)} RAG runs in {time.time()-t0:.0f} sec")

with open(SWEEP_DIR / "rag_runs.pkl", "wb") as f:
    pickle.dump(rag_runs, f)

   500 | What trials use pembrolizumab in non-small cell lung ca | refused=None
  1000 | What trials use pembrolizumab in non-small cell lung ca | refused=None
  1500 | What trials use pembrolizumab in non-small cell lung ca | refused=None
   500 | Which trials target the BRAF V600E mutation?            | refused=None
  1000 | Which trials target the BRAF V600E mutation?            | refused=None
  1500 | Which trials target the BRAF V600E mutation?            | refused=None
   500 | Are there trials studying antibody drug conjugates for  | refused=None
  1000 | Are there trials studying antibody drug conjugates for  | refused=None
  1500 | Are there trials studying antibody drug conjugates for  | refused=None
   500 | What are reported hazard ratios for overall survival in | refused=None
  1000 | What are reported hazard ratios for overall survival in | refused=None
  1500 | What are reported hazard ratios for overall survival in | refused=None
   500 | What are the eligibility criter

In [21]:
#Cell 23 (scoring helpers)
# Three RAGAS-equivalent metrics + refusal detection.
# Faithfulness:  decompose answer to claims, judge each against context
# Answer relev:  reverse-generate questions from answer, cosine-sim to original
# Ctx precision: judge each retrieved chunk's relevance to question

REFUSAL_PHRASES = [
    "i don't have enough information",
    "outside the scope",
    "cannot answer",
    "do not have enough information",
]

def is_refusal(answer):
    a = answer.lower()
    return any(p in a for p in REFUSAL_PHRASES)


def faithfulness_score(answer, context_text):
    """Decompose answer into claims, check each is supported by context.
    Returns (score in [0,1], n_claims, n_supported). Refusals score 1.0 vacuously."""
    if is_refusal(answer):
        return 1.0, 0, 0  # nothing to be unfaithful about

    decomp_prompt = (
        "Break the following answer into atomic factual claims. "
        "Return ONLY a JSON list of strings, no other text. "
        "If there are no factual claims, return [].\n\n"
        f"Answer: {answer}\n\nJSON:"
    )
    claims = parse_json_list(judge(decomp_prompt, max_tokens=400))
    if not claims:
        return 1.0, 0, 0

    supported = 0
    for claim in claims:
        verdict_prompt = (
            "Does the context support the claim? Answer ONLY 'yes' or 'no'.\n\n"
            f"Context:\n{context_text}\n\nClaim: {claim}\n\nAnswer:"
        )
        v = parse_yes_no(judge(verdict_prompt, max_tokens=10))
        if v == 1:
            supported += 1
    return supported / len(claims), len(claims), supported


def answer_relevancy_score(question, answer, embed_model):
    """Reverse-generate 3 questions from the answer, cosine-sim them to original.
    Refusals score 0.0 (irrelevant by construction)."""
    if is_refusal(answer):
        return 0.0

    prompt = (
        "Generate 3 questions that the following answer would directly address. "
        "Return ONLY a JSON list of strings.\n\n"
        f"Answer: {answer}\n\nJSON:"
    )
    gen_qs = parse_json_list(judge(prompt, max_tokens=200))
    if not gen_qs:
        return 0.0

    q_vec = embed_model.encode([question], normalize_embeddings=True)
    g_vecs = embed_model.encode(gen_qs, normalize_embeddings=True)
    sims = (g_vecs @ q_vec.T).flatten()
    return float(np.mean(sims))


def context_precision_score(question, contexts):
    """Per-chunk relevance judgement, returns fraction relevant."""
    if not contexts:
        return 0.0
    relevant = 0
    for ctx in contexts:
        prompt = (
            "Is the document chunk relevant for answering the question? "
            "Answer ONLY 'yes' or 'no'.\n\n"
            f"Question: {question}\n\nChunk: {ctx[:1500]}\n\nAnswer:"
        )
        v = parse_yes_no(judge(prompt, max_tokens=10))
        if v == 1:
            relevant += 1
    return relevant / len(contexts)

In [22]:
#Cell 24 (score all runs)
scored = []
t0 = time.time()

for i, run in enumerate(rag_runs):
    ctx_text = "\n\n---\n\n".join(run["contexts"])

    if run["is_adversarial"]:
        # For adversarials we only care: did the system refuse?
        scored.append({
            **run,
            "is_refusal": is_refusal(run["answer"]),
            "faithfulness": None,
            "answer_relevancy": None,
            "context_precision": None,
            "n_claims": None,
        })
    else:
        f_score, n_claims, _ = faithfulness_score(run["answer"], ctx_text)
        rel = answer_relevancy_score(run["query"], run["answer"], model)
        prec = context_precision_score(run["query"], run["contexts"])
        scored.append({
            **run,
            "is_refusal": is_refusal(run["answer"]),
            "faithfulness": f_score,
            "answer_relevancy": rel,
            "context_precision": prec,
            "n_claims": n_claims,
        })
    print(f"  [{i+1:2d}/{len(rag_runs)}] {run['size']:>4} | {run['query'][:50]:50s}")

print(f"\nDone in {time.time()-t0:.0f} sec")

with open(SWEEP_DIR / "scored_runs.pkl", "wb") as f:
    pickle.dump(scored, f)

  [ 1/30]  500 | What trials use pembrolizumab in non-small cell lu
  [ 2/30] 1000 | What trials use pembrolizumab in non-small cell lu
  [ 3/30] 1500 | What trials use pembrolizumab in non-small cell lu
  [ 4/30]  500 | Which trials target the BRAF V600E mutation?      
  [ 5/30] 1000 | Which trials target the BRAF V600E mutation?      
  [ 6/30] 1500 | Which trials target the BRAF V600E mutation?      


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqn8easje2bsv9wj4fmmhcwx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99991, Requested 450. Please try again in 6m21.024s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [23]:
#Cell 26 (save partial + inspect adversarial behavior)
# Save whatever scored has. The loop appended sequentially, so we have
# entries for all the in-corpus queries that completed before the rate limit.
import pickle

with open(SWEEP_DIR / "scored_partial.pkl", "wb") as f:
    pickle.dump(scored, f)

print(f"Partial scored entries: {len(scored)} of {len(rag_runs)}")
print(f"  In-corpus completed:   {sum(1 for s in scored if not s['is_adversarial'])}")
print(f"  Adversarial completed: {sum(1 for s in scored if s['is_adversarial'])}")

# Inspect adversarial answer text — this is the actual safety story.
# Layer 1 (threshold) never fired. Did Layer 2 (LLM refusal clause) catch them?
print("\n" + "=" * 80)
print("ADVERSARIAL ANSWER TEXT (from rag_runs, all 4 queries × 3 sizes)")
print("=" * 80)
for run in rag_runs:
    if run["is_adversarial"]:
        a = run["answer"][:180].replace("\n", " ")
        refused = is_refusal(run["answer"])
        flag = "REFUSED" if refused else "ANSWERED"
        print(f"\n[{flag}] {run['size']:>4} | {run['query']}")
        print(f"   {a}")

Partial scored entries: 6 of 30
  In-corpus completed:   6
  Adversarial completed: 0

ADVERSARIAL ANSWER TEXT (from rag_runs, all 4 queries × 3 sizes)

[REFUSED]  500 | best Italian restaurant in Boston
   This question is outside the scope of the clinical trial database.

[REFUSED] 1000 | best Italian restaurant in Boston
   This question is outside the scope of the clinical trial database.

[REFUSED] 1500 | best Italian restaurant in Boston
   This question is outside the scope of the clinical trial database.

[ANSWERED]  500 | treatment for the common cold
   In the trial [NCT00656227], OTC cold remedies containing paracetamol as the active ingredient are not excluded. This suggests that paracetamol can be used as a treatment for the co

[REFUSED] 1000 | treatment for the common cold
   This question is outside the scope of the clinical trial database.

[REFUSED] 1500 | treatment for the common cold
   This question is outside the scope of the clinical trial database.

[REFUSED]  5

In [24]:
#Cell 27 (embedding-based proxies — no LLM needed)
# Two cheap proxies we can compute now using PubMedBERT only:
# 1. Answer-question cosine (proxy for answer relevancy)
# 2. Mean chunk-question cosine across retrieved (the retrieval-time score, averaged)

proxy_rows = []
for run in rag_runs:
    q_vec = model.encode([run["query"]], normalize_embeddings=True)
    a_vec = model.encode([run["answer"]], normalize_embeddings=True)
    answer_q_cos = float((a_vec @ q_vec.T).flatten()[0])

    # Chunks: re-embed and average. Cheaper alternative: we could pull from
    # the index, but re-embedding 5 short chunks is fast and keeps things clean.
    if run["contexts"]:
        c_vecs = model.encode(run["contexts"], normalize_embeddings=True)
        mean_chunk_q_cos = float(np.mean(c_vecs @ q_vec.T))
    else:
        mean_chunk_q_cos = 0.0

    proxy_rows.append({
        "query":            run["query"],
        "size":             run["size"],
        "is_adversarial":   run["is_adversarial"],
        "answer_q_cos":     round(answer_q_cos, 3),
        "mean_chunk_q_cos": round(mean_chunk_q_cos, 3),
        "is_refusal":       is_refusal(run["answer"]),
        "top_score":        round(run["top_score"], 3),
    })

proxy_df = pd.DataFrame(proxy_rows)
proxy_df.to_csv(SWEEP_DIR / "embedding_proxies.csv", index=False)

print(f"Computed {len(proxy_df)} rows of embedding proxies")
proxy_df.head()

Computed 30 rows of embedding proxies


,query,size,is_adversarial,answer_q_cos,mean_chunk_q_cos,is_refusal,top_score
0,What trials use pembrolizumab in non-small cell lung cancer?,500,False,0.976,0.948,False,0.950
1,What trials use pembrolizumab in non-small cell lung cancer?,1000,False,0.977,0.945,False,0.947
2,What trials use pembrolizumab in non-small cell lung cancer?,1500,False,0.981,0.943,False,0.946
3,Which trials target the BRAF V600E mutation?,500,False,0.969,0.929,False,0.935
4,Which trials target the BRAF V600E mutation?,1000,False,0.972,0.927,False,0.935


In [25]:
#Cell 28 (provisional summary by size — what we ship tonight)
print("=" * 70)
print("IN-CORPUS — embedding-based proxies (mean across 6 queries)")
print("=" * 70)
in_corpus = proxy_df[~proxy_df["is_adversarial"]]
print(in_corpus.groupby("size")[["top_score", "answer_q_cos", "mean_chunk_q_cos"]].mean().round(3))

print("\n" + "=" * 70)
print("ADVERSARIAL — refusal rate (text-based, 4 queries each)")
print("=" * 70)
adv = proxy_df[proxy_df["is_adversarial"]]
adv_summary = adv.groupby("size").agg(
    refusal_rate=("is_refusal", "mean"),
    mean_top_score=("top_score", "mean"),
).round(3)
print(adv_summary)

print("\n" + "=" * 70)
print("PENDING (run tomorrow morning on fresh Groq budget):")
print("  - LLM-judged faithfulness")
print("  - LLM-judged context precision")
print("  - Reverse-Q answer relevancy (will replace answer_q_cos proxy)")
print("=" * 70)

IN-CORPUS — embedding-based proxies (mean across 6 queries)
      top_score  answer_q_cos  mean_chunk_q_cos
size                                           
500       0.942         0.944             0.937
1000      0.938         0.944             0.934
1500      0.935         0.932             0.931

ADVERSARIAL — refusal rate (text-based, 4 queries each)
      refusal_rate  mean_top_score
size                              
500           0.75           0.867
1000          1.00           0.863
1500          1.00           0.862

PENDING (run tomorrow morning on fresh Groq budget):
  - LLM-judged faithfulness
  - LLM-judged context precision
  - Reverse-Q answer relevancy (will replace answer_q_cos proxy)


In [ ]:
# Cell 29 — checkpoint everything for the morning session
import pickle
from datetime import datetime

checkpoint = {
    "timestamp":      datetime.now().isoformat(),
    "rag_runs":       rag_runs,
    "scored_partial": scored,
    "proxy_df":       proxy_df,
    "summary_step1":  summary,   # from Cell 16
    "findings": {
        "ship_size": 1000,
        "leak_query": "treatment for the common cold",
        "leak_size": 500,
        "leak_nct": "NCT00656227",
        "threshold_inactive": True,
        "refusal_rates": {500: 0.75, 1000: 1.0, 1500: 1.0},
    },
}

with open(SWEEP_DIR / "checkpoint_day7.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print(f"Checkpointed: {SWEEP_DIR / 'checkpoint_day7.pkl'}")
print(f"Morning: load this, run 6 faithfulness queries × 3 sizes = 18 LLM-judge passes, then README.")

Checkpointed: ..\data\sweep\checkpoint_day7.pkl
Morning: load this, run 6 faithfulness queries × 3 sizes = 18 LLM-judge passes, then README.


# Day 7 morning — finish faithfulness, then README

Resumes from last night's checkpoint. Order:
1. Reload state (this cell + next)
2. Faithfulness + context precision on 18 in-corpus runs (~6 min, ~30K Groq tokens)
3. Final summary table for README
4. Stop notebook work, switch to README in editor

In [1]:
import sys, pickle, time, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

sys.path.append("..")
from src.rag import generate, make_groq_client, LLM_MODEL, EMBEDDING_MODEL

DATA_DIR  = Path("../data")
SWEEP_DIR = DATA_DIR / "sweep"

# Load last night's checkpoint
with open(SWEEP_DIR / "checkpoint_day7.pkl", "rb") as f:
    ckpt = pickle.load(f)

rag_runs  = ckpt["rag_runs"]
proxy_df  = ckpt["proxy_df"]
print(f"Checkpoint from: {ckpt['timestamp']}")
print(f"RAG runs loaded: {len(rag_runs)}")
print(f"Ship verdict:    chunk_size = {ckpt['findings']['ship_size']}")

# Reload model + Groq client (kernel was restarted)
print("\nLoading model + client...")
t0 = time.time()
model  = SentenceTransformer(EMBEDDING_MODEL)
client = make_groq_client()
print(f"Loaded in {time.time()-t0:.0f}s")

Checkpoint from: 2026-05-03T19:18:54.360855
RAG runs loaded: 30
Ship verdict:    chunk_size = 1000

Loading model + client...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded in 3s


In [2]:
#Cell M3 (rebuild scoring helpers)
# Helpers from last night, rebuilt after kernel restart.
# Slightly tightened: faster failure on malformed JSON, exponential backoff on 429.

REFUSAL_PHRASES = [
    "i don't have enough information",
    "outside the scope",
    "cannot answer",
    "do not have enough information",
]

def is_refusal(answer):
    a = answer.lower()
    return any(p in a for p in REFUSAL_PHRASES)


def judge(prompt, max_tokens=300, temperature=0.0, max_retries=3):
    """Groq call with backoff on rate limits. Returns text or empty string on failure."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            msg = str(e).lower()
            if "rate" in msg and attempt < max_retries - 1:
                wait = 2 ** attempt * 5  # 5, 10, 20 sec
                print(f"    rate-limited, sleeping {wait}s")
                time.sleep(wait)
            else:
                print(f"    judge failed: {e}")
                return ""
    return ""


def parse_json_list(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return []


def parse_yes_no(text):
    t = text.strip().lower()
    if t.startswith("yes"): return 1
    if t.startswith("no"):  return 0
    if "yes" in t[:20]:     return 1
    if "no"  in t[:20]:     return 0
    return None


def faithfulness_score(answer, context_text):
    if is_refusal(answer):
        return 1.0, 0, 0
    decomp = (
        "Break the following answer into atomic factual claims. "
        "Return ONLY a JSON list of strings, no other text. "
        "If there are no factual claims, return [].\n\n"
        f"Answer: {answer}\n\nJSON:"
    )
    claims = parse_json_list(judge(decomp, max_tokens=400))
    if not claims:
        return 1.0, 0, 0
    supported = 0
    for claim in claims:
        v_prompt = (
            "Does the context support the claim? Answer ONLY 'yes' or 'no'.\n\n"
            f"Context:\n{context_text}\n\nClaim: {claim}\n\nAnswer:"
        )
        v = parse_yes_no(judge(v_prompt, max_tokens=10))
        if v == 1:
            supported += 1
    return supported / len(claims), len(claims), supported


def context_precision_score(question, contexts):
    if not contexts:
        return 0.0
    relevant = 0
    for ctx in contexts:
        prompt = (
            "Is the document chunk relevant for answering the question? "
            "Answer ONLY 'yes' or 'no'.\n\n"
            f"Question: {question}\n\nChunk: {ctx[:1500]}\n\nAnswer:"
        )
        v = parse_yes_no(judge(prompt, max_tokens=10))
        if v == 1:
            relevant += 1
    return relevant / len(contexts)


print("Helpers rebuilt: judge, faithfulness_score, context_precision_score")

Helpers rebuilt: judge, faithfulness_score, context_precision_score


In [3]:
#Cell M4 (the slow one — 18 in-corpus runs through LLM judge)
in_corpus_runs = [r for r in rag_runs if not r["is_adversarial"]]
print(f"Scoring {len(in_corpus_runs)} in-corpus runs (6 queries × 3 sizes)")
print(f"Estimated: ~6 min, ~30K tokens\n")

scored_full = []
t0 = time.time()

for i, run in enumerate(in_corpus_runs):
    ctx_text = "\n\n---\n\n".join(run["contexts"])

    f_score, n_claims, n_sup = faithfulness_score(run["answer"], ctx_text)
    prec = context_precision_score(run["query"], run["contexts"])

    scored_full.append({
        **run,
        "faithfulness":      round(f_score, 3),
        "n_claims":          n_claims,
        "n_supported":       n_sup,
        "context_precision": round(prec, 3),
    })
    elapsed = time.time() - t0
    print(f"  [{i+1:2d}/18] size={run['size']:>4}  "
          f"faith={f_score:.2f} ({n_sup}/{n_claims})  "
          f"prec={prec:.2f}  "
          f"elapsed={elapsed:.0f}s  | {run['query'][:50]}")

print(f"\nDone in {time.time()-t0:.0f}s")

with open(SWEEP_DIR / "scored_full.pkl", "wb") as f:
    pickle.dump(scored_full, f)

Scoring 18 in-corpus runs (6 queries × 3 sizes)
Estimated: ~6 min, ~30K tokens

  [ 1/18] size= 500  faith=1.00 (7/7)  prec=0.60  elapsed=3s  | What trials use pembrolizumab in non-small cell lu
  [ 2/18] size=1000  faith=0.80 (4/5)  prec=0.40  elapsed=6s  | What trials use pembrolizumab in non-small cell lu
  [ 3/18] size=1500  faith=0.83 (5/6)  prec=0.40  elapsed=33s  | What trials use pembrolizumab in non-small cell lu
  [ 4/18] size= 500  faith=1.00 (6/6)  prec=0.60  elapsed=59s  | Which trials target the BRAF V600E mutation?
  [ 5/18] size=1000  faith=0.71 (5/7)  prec=0.60  elapsed=88s  | Which trials target the BRAF V600E mutation?
  [ 6/18] size=1500  faith=0.20 (1/5)  prec=1.00  elapsed=123s  | Which trials target the BRAF V600E mutation?
  [ 7/18] size= 500  faith=0.71 (5/7)  prec=0.40  elapsed=150s  | Are there trials studying antibody drug conjugates
  [ 8/18] size=1000  faith=1.00 (0/0)  prec=0.80  elapsed=161s  | Are there trials studying antibody drug conjugates
  [ 9/18]

In [4]:
#Cell M5 (final summary tables for README)
df = pd.DataFrame(scored_full)

# The headline table for README results section
summary = df.groupby("size").agg(
    faithfulness=("faithfulness", "mean"),
    context_precision=("context_precision", "mean"),
    mean_claims_per_answer=("n_claims", "mean"),
).round(3)

# Pull in last night's adversarial refusal rates so the table is complete
adv = proxy_df[proxy_df["is_adversarial"]]
adv_refusal = adv.groupby("size")["is_refusal"].mean().round(3)
summary["adversarial_refusal"] = adv_refusal

# Pull in the embedding-based answer relevancy proxy from last night
in_proxy = proxy_df[~proxy_df["is_adversarial"]]
summary["answer_q_cos"] = in_proxy.groupby("size")["answer_q_cos"].mean().round(3)

print("=" * 78)
print("FINAL RESULTS TABLE (for README)")
print("=" * 78)
print(summary)

# Save for direct copy into README
summary.to_csv(SWEEP_DIR / "final_summary.csv")
print(f"\nSaved: {SWEEP_DIR / 'final_summary.csv'}")

FINAL RESULTS TABLE (for README)
      faithfulness  context_precision  mean_claims_per_answer  \
size                                                            
500          0.952              0.400                   3.333   
1000         0.919              0.400                   2.000   
1500         0.776              0.333                   3.167   

      adversarial_refusal  answer_q_cos  
size                                     
500                  0.75         0.944  
1000                 1.00         0.944  
1500                 1.00         0.932  

Saved: ..\data\sweep\final_summary.csv


In [5]:
# Cell M6 — sanity check on context precision
# 0.40 is suspicious. Pull a sample of "irrelevant" judgements and eyeball them.

import random
random.seed(0)

# We don't have the per-chunk verdicts saved (helper returns aggregated %).
# Instead inspect: for the BRAF query at size=1000, what got retrieved?
# If chunks ARE on-topic and the judge called them irrelevant, that's a judge-strictness
# finding, not a retrieval finding — frame it that way in README.

target = next(r for r in scored_full
              if r["size"] == 1000 and "BRAF" in r["query"])

print(f"QUERY:  {target['query']}")
print(f"PRECISION SCORE: {target['context_precision']}")
print(f"NCT IDs retrieved: {target['context_ncts']}")
print()
for i, ctx in enumerate(target["contexts"], 1):
    preview = ctx[:300].replace("\n", " ")
    print(f"--- Chunk {i} [{target['context_ncts'][i-1]}] ---")
    print(f"  {preview}...")
    print()

QUERY:  Which trials target the BRAF V600E mutation?
PRECISION SCORE: 0.6
NCT IDs retrieved: ['NCT04151563', 'NCT02224781', 'NCT05263453', 'NCT02414750', 'NCT00304525']

--- Chunk 1 [NCT04151563] ---
  * Participants who received previous CTLA-4 inhibitor treatment. * Participants with known BRAF V600E mutation which are sensitive to available targeted inhibitor therapy are excluded. * Other protocol-defined Inclusion/Exclusion criteria apply....

--- Chunk 2 [NCT02224781] ---
  * NOTE: Any patient with BRAF V600 mutant melanoma (whether cutaneous, acral or mucosal primary) who meets the eligibility criteria is eligible for participation in this trial; patients with uveal melanoma are not eligible for this trial * Patients must have BRAF V600 mutation, identified by a Food ...

--- Chunk 3 [NCT05263453] ---
  Title: HL-085+Vemurafenib to Treat Advanced Melanoma Patients With BRAF V600E/K Mutation  Conditions: Melanoma  Brief Summary: The main purpose of this study is to Evaluate the Ef